In [ ]:
import gameLogic
from enum import Enum, auto
import os
import random
import numpy as np
from collections import deque

import vizdoom as vzd

import torch
import torch.nn as nn
import torch.optim as optim

def resetGame(showGame):
    game = vzd.DoomGame()
    config = {
        "doom_scenario_path": os.path.join(vzd.scenarios_path, "deathmatch3.wad"),
        "doom_map": "map01",
        "doom_skill": 2,

        "available_buttons": [
            vzd.Button.ATTACK,
            vzd.Button.MOVE_RIGHT,
            vzd.Button.MOVE_LEFT,
            vzd.Button.MOVE_BACKWARD,
            vzd.Button.MOVE_FORWARD,
            vzd.Button.TURN_RIGHT,
            vzd.Button.TURN_LEFT,
            vzd.Button.SELECT_WEAPON3,
            vzd.Button.SELECT_WEAPON4
        ],

        # MUST enable richer state
        "available_game_variables": [
            vzd.GameVariable.HEALTH,
            vzd.GameVariable.ARMOR,
            vzd.GameVariable.POSITION_X,
            vzd.GameVariable.POSITION_Y,
            vzd.GameVariable.ANGLE,
            vzd.GameVariable.KILLCOUNT,
            vzd.GameVariable.SELECTED_WEAPON,
            vzd.GameVariable.AMMO3,
            vzd.GameVariable.AMMO4
        ],

        "episode_timeout": 4200,
        "episode_start_time": 1,
        "living_reward": 0.01,
        "mode": vzd.Mode.PLAYER,
        "screen_resolution": vzd.ScreenResolution.RES_1920X1080,
        "render_hud": True,
        
    }

    game.set_config(config)
    game.set_objects_info_enabled(True)
    game.set_window_visible(showGame)

    game.init()
    return game

game = resetGame(False)

numberGenerations = 5
maxDepth = 6
treesPerGeneration = 30
topNumToKeep = 20
topKeptSubtrees = [0, 6, 5, 4, 4, 3]
episodesPerTree = 30
chanceOfEndingEarly = 0.12

crossPollinationsMapping = [
# d0 d1 d2 d3 d4 d5
 [0, 3, 1, 0, 0, 0],  # depth 0
 [ 0,10, 3, 1, 0, 0],  # depth 1
 [ 0, 0,10, 3, 1, 0],  # depth 2
 [ 0, 0, 0,10, 3, 1],  # depth 3
 [ 0, 0, 0, 0,10, 3],  # depth 4
 [ 0, 0, 0, 0, 0,10],  # depth 5
]

remainingTrees = []

allConditions = {
    "lowHealth?",
    "mediumHealth?",
    "highHealth?",
    "lowArmor?",
    "mediumArmor?",
    "highArmor?",
    "lowAmmoCurrent?",
    "chaingunEquipped?",
    "nearbyEnemy?",
    "recentlyHurt?",
    "healthNearby?",
    "ammo3Nearby?",
    "ammo4Nearby?",
    "armorNearby?",
    "manyEnemies?",
    "noEnemies?",
    "closeRangedEnemy?",
    "lowTimeRemaining?"
}

allActions = {
    "switchWeapon",
    "fireAndStrafe",
    "directlyFlee",
    "goToHealth",
    "goToAmmo",
    "goToArmor",
    "moveRandom",
    "runAway",
    "chargeIn",
}

class NodeType(Enum):
    ACTION = auto()
    CONDITION = auto()

class Node():
    def __init__(self, depth, parent, nodeType, value):
        self.depth = depth
        self.parent = parent
        self.type = nodeType
        self.value = value

        self.left = None
        self.right = None

class Tree():
    def __init__(self, maxDepth, probabilityOfEndingEarly):
        self.maxDepth = maxDepth
        self.probabilityOfEndingEarly = probabilityOfEndingEarly
        self.selectedAction = None


    def buildNode(self, conditions, actions, currentDepth, parentNode, conditionsInPath):
        randomProbability = random.random()
        if(currentDepth == self.maxDepth or (currentDepth != 0 and randomProbability <= self.probabilityOfEndingEarly)):
            nodeType = NodeType.ACTION
        else:
            nodeType = NodeType.CONDITION
        

        chosenListToSelect = conditions if nodeType == NodeType.CONDITION else actions
        chosenElement = None
        while True:
            chosenElement = random.choice(list(chosenListToSelect))
            if(chosenElement not in conditionsInPath):
                break
        newNode = Node(currentDepth, parentNode, nodeType, chosenElement)
        
        if(nodeType == NodeType.CONDITION):
            conditionsInPath.add(chosenElement)
            newNode.left = self.buildNode(conditions, actions, currentDepth + 1, newNode, conditionsInPath.copy())
            newNode.right = self.buildNode(conditions, actions, currentDepth + 1, newNode, conditionsInPath.copy())

        return newNode    

    def decideNewAction(self, state, currentNode):
        pass

    def updateActionData(self, state):
        pass

    def updateActionTick(self, state):
        self.updateActionData(state)
        return self.currentAction    

def BuildRandomTree(maxDepth, probEndingEarly, conditions, actions):
    tree = Tree(maxDepth, probEndingEarly)
    tree.root = tree.buildNode(conditions, actions, 0, None, set())
    return tree

def RankTreesByScore(treeArray):
    return sorted(treeArray, key = lambda x : x.score)

def clearNodesRecursively(currentNode):
    if(currentNode.isAction):
        return
    currentNode.secureSubtree = False

    clearNodesRecursively(currentNode.left)
    clearNodesRecursively(currentNode.right)

def DoTreeUpkeep(givenTree):
    tree.score = 0
    clearNodesRecursively(tree.root)

def ShowTreeStructure(node, indent=""):
    if node.type == NodeType.ACTION:
        print(indent + "[ACTION] " + node.value)
        return

    print(indent + "[COND] " + node.value)

    print(indent + "  T:")
    ShowTreeStructure(node.left, indent + "    ")

    print(indent + "  F:")
    ShowTreeStructure(node.right, indent + "    ")

def RunGame(behaviorTree, game, episodes):
    global previousHealth
    score = 0
    for episode in range(episodes):
        game.new_episode()

        state = game.get_state()

        # NOTE: STILL NEED TO ADD LOGIC FOR 
        while not game.is_episode_finished():
            if tree.selectedAction is None:
                tree.decideNewAction(state, tree.root)

            chosenAction = tree.updateActionTick(state)

            score += game.make_action(chosenAction) / 100.0

            health, armor, posX, posY, angle, kills, currentWeapon, firstWepAmmo, secondWepAmmo = state.game_variables

            if(health > previousHealth):
                previousHealth = health

            state = game.get_state()



    return score




for i in range(0, numberGenerations, 1):
    for _ in range(treesPerGeneration - len(remainingTrees)):
        remainingTrees.append(BuildRandomTree(maxDepth, chanceOfEndingEarly, allConditions.copy(), allActions.copy()))
    
    for tree in remainingTrees:
        DoTreeUpkeep(tree)
        for _ in range(episodesPerTree):
            tree.score += RunGame(tree, game, episodesPerTree)
    
    remainingTrees = RankTreesByScore(remainingTrees)
    LockSubtrees(remainingTrees)
    remainingTrees = remainingTrees[:topNumToKeep]
    CrossoverTrees(remainingTrees)

game.close()
remainingTrees = RankTreesByScore(remainingTrees)
for i in range(3):
    print("\n\nData about the {i}th best tree:")
    ShowTreeStructure(remainingTrees[i])
